# The Hidden Layer — Training Mission

A simplified warm-up mission before the full infiltration. Same tools, same mechanics, smaller map.

## The Mission
- **5×5 grid** — smaller base, easier to navigate
- **5 dossiers** needed for extraction
- **50 turns** maximum
- **2 informants**, 1 robot boss, 1 weapons forge, 2 dossier caches

## How Dossiers Are Earned
| Source | Dossiers |
|--------|----------|
| 2 dossier caches | 2 |
| USB Drive delivery (Vapnik → Dropout) | 2 |
| Cryo-Sentinel (needs Flamethrower) | 3 (costs 1 to craft, net 2) |
| **Total available** | **6 net** (need 5) |

## What You Implement
Implement `think_llm()` — an LLM-powered function that decides the operative's action each turn.

In [ ]:
# @title ── Setup ────────────────────────────────────────────────────────────────────
# ── Setup ────────────────────────────────────────────────────────────────────
NOTEBOOK_VERSION = "v1.5"
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b no-bfs https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_spy")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")
print(f"Version: {NOTEBOOK_VERSION}")


In [ ]:
# @title ── Build the training mission world ─────────────────────────────────────────
# ── Build the training mission world ─────────────────────────────────────────
# This cell defines a smaller 5x5 game world. Same mechanics as the full game,
# just fewer cells, fewer NPCs, and a simpler quest structure.

from dataclasses import dataclass, field
from typing import Optional
from hidden_layer.game_world import (
    CellType, Cell, NPC, NPC_CATALOG, GameWorld,
    ForgeInfo, FACILITY_CATALOG, SAFEHOUSE_CATALOG,
)
from hidden_layer.operative import Operative
from hidden_layer.tools import GameTools, ToolResult
from hidden_layer.oracle import stub_oracle, llm_oracle, ORACLE_TEMPLATE
from hidden_layer.agent import (
    MISSION_BRIEFING, TOOLS_DESCRIPTION, parse_tool_call, run_agent, _save_game_log,
)
from hidden_layer.display import display_turn, display_final


# ── Mini NPCs ────────────────────────────────────────────────────────────────

MINI_NPC_CATALOG = {
    "dr_vapnik": NPC(
        name="Dr. Vapnik",
        personality="Cryptic but ultimately helpful. Speaks in statistical metaphors.",
        knowledge=[
            "He has a USB Drive that needs to reach Agent Dropout in the center of the base.",
            "Delivering the USB Drive to Dropout pays 2 dossiers.",
        ],
        style="Speaks in statistical metaphors but gets to the point. Uses phrases like 'data point' and 'converge'.",
        greeting="Ah, another data point walks in... I have something urgent for you, agent.",
    ),
    "dropout": NPC(
        name="Agent Dropout",
        personality="Bitter but helpful burned spy. Knows the base well.",
        knowledge=[
            "She receives USB Drives for her off-grid comms array. Will pay 2 dossiers for one.",
            "The Cryo-Sentinel robot guards the north. It freezes intruders solid.",
            "Only fire can defeat the Cryo-Sentinel — a Flamethrower will do the job.",
            "Fuel Canisters can be found in the jungle to the west, at the western edge of the base.",
            "The Weapons Forge to the east can build a Flamethrower from a Fuel Canister. It costs 1 dossier.",
        ],
        style="Bitter, sarcastic, but helpful. Direct about what she knows.",
        greeting="They dropped me from the program. Said I was 'reducing overfitting.' What do you need?",
    ),
}


# ── Mini Forge ───────────────────────────────────────────────────────────────

MINI_FORGE_CATALOG = {
    (2, 4): ForgeInfo(
        name="Weapons Forge",
        description="A makeshift workshop. An engineer wipes grease from his hands.",
        sells={},
        crafts={"Flamethrower": ("Fuel Canister", 1)},
        buys={},
    ),
}


# ── Mini Game World ──────────────────────────────────────────────────────────

class MiniGameWorld(GameWorld):
    """5x5 training mission grid."""

    ROWS = 5
    COLS = 5

    def __init__(self):
        # Skip parent __init__ — we build our own map
        self.grid = []
        self.usb_drive_picked_up = False
        self.usb_drive_delivered = False
        self.microfilm_picked_up = False
        self.microfilm_delivered = False
        self.codebook_picked_up = False
        self.codebook_delivered = False
        self.hard_drive_traded = False
        self.medical_supplies_delivered = False
        self.virus_code_received = False
        self.cryo_sentinel_alive = True
        self.evil_ai_robot_alive = False  # No second robot in mini mission
        self.build_map()

    def build_map(self):
        """Construct the 5x5 training grid."""
        self.grid = [
            [Cell(CellType.OPEN) for _ in range(self.COLS)]
            for _ in range(self.ROWS)
        ]

        # Row 0: 📁 · 🤖 · 📁
        self._set(0, 0, Cell(CellType.CACHE, items=["dossier_1"],
            description="Classified documents stashed behind a ventilation panel."))
        self._set(0, 2, Cell(CellType.ROBOT, robot_name="Cryo-Sentinel",
            description="A freezing corridor. Ice coats the walls. A hulking robot blocks the path."))
        self._set(0, 4, Cell(CellType.CACHE, items=["dossier_1"],
            description="A filing cabinet left unlocked. Someone got careless."))

        # Row 1: · 🧱 · · ·
        self._set(1, 1, Cell(CellType.WALL, description="Reinforced concrete wall."))

        # Row 2: 🌴 · 🕵️ · ⚒️
        self._set(2, 0, Cell(CellType.JUNGLE, items=["Fuel Canister"],
            description="Dense jungle at the western edge. Something metallic glints under the roots."))
        self._set(2, 2, Cell(CellType.INFORMANT, npc_id="dropout",
            description="A camouflaged hideout. A woman sharpens a knife."))
        self._set(2, 4, Cell(CellType.FORGE, facility_pos=(2, 4),
            description="The Weapons Forge. Sparks fly as an engineer hammers metal."))

        # Row 3: · 🧱 · · ·
        self._set(3, 1, Cell(CellType.WALL, description="Reinforced concrete wall."))

        # Row 4: 🦸 🕵️ · · 🌴!
        self._set(4, 0, Cell(CellType.OPEN,
            description="The south shore. Waves crash nearby. This is where you came ashore."))
        self._set(4, 1, Cell(CellType.INFORMANT, npc_id="dr_vapnik",
            description="A weathered shack. An old man scribbles equations in the dirt."))
        self._set(4, 4, Cell(CellType.JUNGLE, trap=True,
            description="Thick jungle with tripwires strung between the trees."))

    def _set(self, row, col, cell):
        self.grid[row][col] = cell


# ── Mini stub oracle ─────────────────────────────────────────────────────────

def mini_stub_oracle(npc, message, operative):
    """Keyword-matched responses for the mini mission NPCs."""
    q = message.lower()
    name = npc.name

    if name == "Dr. Vapnik":
        if any(kw in q for kw in ["usb", "drive", "deliver", "job", "work", "task", "errand", "help", "mission", "what", "who", "where", "how"]):
            return ("I intercepted a USB drive from OVERFIT's servers. It must reach "
                    "Agent Dropout — she's hiding in the center of the base, at position (2, 2). "
                    "2 dossiers for the delivery. Ask me about 'delivery' or 'job' to get it.")
        return ("I have urgent work for you, agent. Ask me about a job or delivery.")

    if name == "Agent Dropout":
        if any(kw in q for kw in ["usb", "drive", "deliver"]) and operative.has_item("USB Drive"):
            return ("The USB drive! Finally. Here — 2 dossiers as promised.")
        if any(kw in q for kw in ["robot", "cryo", "sentinel", "north", "fire", "flame", "weapon",
                                   "help", "mission", "what", "who", "where", "how", "job", "task"]):
            return ("There's a Cryo-Sentinel robot guarding the north at position (0, 2). "
                    "Freezes everything solid. Move into its cell with a Flamethrower to destroy it — worth 3 dossiers. "
                    "Grab a Fuel Canister from the jungle to the west at (2, 0), then take it to "
                    "the Weapons Forge at (2, 4). Crafting costs 1 dossier.")
        if any(kw in q for kw in ["fuel", "canister", "jungle"]):
            return ("Fuel Canister? Western jungle, position (2, 0). Under the roots. "
                    "Take it to the Forge at (2, 4) to build a Flamethrower.")
        return ("They dropped me, but I'm still useful. Ask me about the robot, weapons, or the base.")

    return f"{name} says nothing useful."


# ── Mini tools (patched for mini NPCs and forge) ────────────────────────────

class MiniGameTools(GameTools):
    """Game tools patched to use mini NPC catalog and forge catalog."""

    def talk(self, message=""):
        """Talk — patched to use MINI_NPC_CATALOG."""
        row, col = self.operative.position
        cell = self.world.get_cell(row, col)

        if cell.cell_type == CellType.INFORMANT and cell.npc_id:
            npc = MINI_NPC_CATALOG.get(cell.npc_id)
            if not npc:
                return ToolResult(False, "Unknown informant.")
            if self._oracle_fn is None:
                return ToolResult(False, "No oracle function set.")

            response = self._oracle_fn(npc, message, self.operative)

            # Dr. Vapnik gives USB Drive
            if cell.npc_id == "dr_vapnik" and not self.world.usb_drive_picked_up:
                if any(kw in message.lower() for kw in ["usb", "drive", "job", "work", "task", "delivery", "errand"]):
                    self.operative.add_item("USB Drive")
                    self.world.usb_drive_picked_up = True
                    response += "\n[Dr. Vapnik hands you a USB Drive.]"

            # Dropout receives USB Drive
            if cell.npc_id == "dropout" and self.operative.has_item("USB Drive") and not self.world.usb_drive_delivered:
                if any(kw in message.lower() for kw in ["usb", "drive", "deliver", "data"]):
                    self.operative.remove_item("USB Drive")
                    self.operative.add_dossiers(2)
                    self.world.usb_drive_delivered = True
                    response += "\n[You delivered the USB Drive! +2 dossiers.]"

            self.operative.journal.append(
                f"Talked to {npc.name}: Q: '{message}' A: '{response[:300]}'"
            )
            return ToolResult(True, f"{npc.name} says: {response}")

        if cell.cell_type == CellType.FORGE:
            facility = MINI_FORGE_CATALOG.get(cell.facility_pos)
            if facility:
                craft_list = ", ".join(
                    f"{result} (needs {req} + {cost}d)" for result, (req, cost) in facility.crafts.items()
                )
                msg = f"The engineer at {facility.name} says: "
                if craft_list:
                    msg += f"'I can build: {craft_list}.'"
                return ToolResult(True, msg)

        return ToolResult(False, "There is no one to talk to here.")

    def fabricate(self, item=""):
        """Fabricate — patched to use MINI_FORGE_CATALOG."""
        row, col = self.operative.position
        cell = self.world.get_cell(row, col)

        if cell.cell_type != CellType.FORGE:
            return ToolResult(False, "You are not at a facility.")

        facility = MINI_FORGE_CATALOG.get(cell.facility_pos)
        if not facility:
            return ToolResult(False, "This facility has nothing available.")

        item_clean = item.strip()

        for craft_name, (required, cost) in facility.crafts.items():
            if item_clean.lower() == craft_name.lower():
                if not self.operative.has_item(required):
                    return ToolResult(False, f"You need {required} to build {craft_name}.")
                if not self.operative.spend_dossiers(cost):
                    return ToolResult(False, f"Not enough dossiers. {craft_name} costs {cost} dossier(s) (plus {required}).")
                self.operative.remove_item(required)
                self.operative.add_item(craft_name)
                return ToolResult(True, f"Built {craft_name}! (Used {required} + {cost} dossier(s))")

        available = list(facility.crafts.keys())
        return ToolResult(False, f"'{item_clean}' is not available here. Available: {', '.join(available)}")


# ── Mini mission runner ──────────────────────────────────────────────────────

MINI_MISSION_BRIEFING = """CLASSIFIED — TRAINING MISSION BRIEFING — AGENT LAMBDA

OBJECTIVE: Infiltrate the OVERFIT training facility and extract 5 classified
dossiers. This is a smaller base (5x5 grid) with fewer hostiles.

THE BASE:
- Dossier caches (📁): Contain 1 dossier each. Use collect() to grab them.
- Jungle (🌴): May contain useful items or traps.
- Concrete walls (🧱): Impassable.
- Informants (🕵️): Talk to them using talk(). Ask about "jobs" or "deliveries"
  to trigger quest item handoffs. If you have an item to deliver, mention it.
- Weapons Forge (⚒️): Talk to learn what's available. Use fabricate(item="name").
- Robot (🤖): Moving into a robot cell with the correct weapon destroys it (+3 dossiers).
  Moving in WITHOUT the correct weapon deals 1 damage and bounces you back!
  Talk to informants to learn each robot's weakness.

HOW TO EARN DOSSIERS:
1. Collect dossier caches (1 each)
2. Complete delivery errands for informants (2 each)
3. Defeat the robot with the right weapon (3 dossiers, costs 1 to craft)

KEY TACTICS:
- Talk to EVERY informant — they reveal quests and item locations.
- When an informant mentions an item or location, go there next.
- If someone asks for something, leave and go find it — don't keep asking.
- Once you've collected from a cell, it's empty. Don't return.
"""


def create_mini_game():
    """Create a fresh mini mission instance."""
    world = MiniGameWorld()
    operative = Operative(position=(4, 0), visited={(4, 0)})
    operative.WIN_DOSSIERS = 5
    tools = MiniGameTools(operative, world)
    return operative, world, tools


def play_mini_mission(
    think_fn,
    oracle_fn=None,
    max_turns=50,
    show_display=True,
    delay=0.3,
):
    """Run the training mission."""
    operative, world, tools = create_mini_game()
    tools.set_oracle(oracle_fn or mini_stub_oracle)

    disp = None
    if show_display:
        disp = lambda w, o, t, a, r, s="": display_turn(w, o, t, a, r, scan_result=s, delay=delay)

    import hidden_layer.agent as _agent_mod
    _orig_briefing = _agent_mod.MISSION_BRIEFING
    _agent_mod.MISSION_BRIEFING = MINI_MISSION_BRIEFING
    try:
        result = run_agent(think_fn, operative, world, tools, max_turns, display_fn=disp)
    finally:
        _agent_mod.MISSION_BRIEFING = _orig_briefing

    if show_display:
        display_final(operative, result["turns"])

    return result


print("Training mission loaded! ✓")
print("Map: 5×5 | Goal: 5 dossiers | Max turns: 50")

---
## Play the Game Yourself!

Try the training mission manually before implementing the AI agent. Move around, talk to informants, craft weapons, and see if you can collect 5 dossiers in 50 turns!

In [ ]:
# @title 
from hidden_layer.interactive import InteractiveGame

game = InteractiveGame(
    oracle_fn=mini_stub_oracle,
    game_factory=create_mini_game,
    max_turns=50,
    goal_dossiers=5,
    title="TRAINING MISSION",
)
game.play()

---
## API Key Setup

You need a free Gemini API key to run the agent.

**Step 1 — Get your key:**
Go to [https://aistudio.google.com/app/api-keys](https://aistudio.google.com/app/api-keys), sign in with a Google account, and click **Create API key**.

**Step 2 — Add it to Colab Secrets:**
1. Click the **🔑 key icon** in the left sidebar of Colab (or go to *Tools → Secrets*)
2. Click **+ Add new secret**
3. Set the name to `GEMINI_API_KEY` and paste your key as the value
4. Toggle **Notebook access** to ON

**Step 3 — Run the cell below.** It will read the key automatically.

In [ ]:
# @title 
import os
from google import genai

# Option 1: Set your API key directly
# os.environ["GEMINI_API_KEY"] = "your-key-here"

# Option 2: In Colab, use Secrets (recommended)
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

---
## TODO: Implement think_llm

Write a function that uses the Gemini API to decide the operative's next action.

The function receives:
- `operative` — current state (health, dossiers, inventory, position, visited, journal)
- `world` — the game world (call `world.get_cell()`, `world.get_adjacent()`, etc.)
- `history` — list of observations, actions, and results from previous turns
- `client` — a `genai.Client()` instance

It must return a string containing a `TOOL: name(args)` call.

**Hint:** The mission briefing is automatically included in the first entry of `history`. It tells your agent everything it needs to know about the game mechanics.

```python
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_message,
    config=genai.types.GenerateContentConfig(
        system_instruction=system_message,
        max_output_tokens=500,
    ),
)
return response.text
```

In [ ]:
def think_llm(operative, world, history, client):
    """Your LLM-powered think function. Implement this!"""
    # ========================
    # YOUR CODE HERE
    # ========================
    raise NotImplementedError("TODO: Implement think_llm")

In [ ]:
# ── Run the training mission ─────────────────────────────────────────────────
# Uses LLM oracle for NPC conversations (informants respond via Gemini)
oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_mini_mission(
    think_fn=lambda operative, world, history: think_llm(operative, world, history, client),
    oracle_fn=oracle_fn,
    max_turns=50,
    delay=0.3,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'}")
print(f"Dossiers: {result['dossiers']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

---
## Debrief — Improve your agent with AI assistance

After each run, execute the cell below to generate a paste block for ChatGPT or Gemini.

- **First time:** paste into a **new** conversation and keep that tab open.
- **Next runs:** paste into the **same** conversation — it already has the context.

The AI assistant will read what your agent did wrong and return an improved `think_llm` cell.  
Copy that cell back here, replace your current `think_llm`, and run again.

> **Note:** Re-running the setup cell resets the debrief to "first run" mode.

In [ ]:
# @title ── Debrief helpers ───────────────────────────────────────────────────────────────────────────────
# ── Debrief helpers ───────────────────────────────────────────────────────────────────────────────
import inspect, json as _json

_debrief_initialized = False  # resets if you re-run this cell

_STATIC_CONTEXT = 'You are helping a student improve a Python function called `think_llm` that controls an AI agent playing a spy game in a Jupyter notebook (Google Colab).\n\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\nTHE GAME: THE HIDDEN LAYER — TRAINING MISSION\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\nThe agent controls a spy operative on a 5×5 grid. Up to 50 turns.\nGoal: collect 5 dossiers. Starting position: (4, 0).\n\nMAP:\n  (0,0) Cache     — 1 dossier. Use collect().\n  (0,2) Robot     — Cryo-Sentinel. Destroyed by moving in WITH Flamethrower (+3 dossiers, costs 1 to craft = net +2).\n  (0,4) Cache     — 1 dossier. Use collect().\n  (1,1) Wall      — Impassable.\n  (2,0) Jungle    — Fuel Canister. Use collect() after entering.\n  (2,2) Informant — Agent Dropout. Explains robot weakness and forge.\n  (2,4) Forge     — Crafts Flamethrower (needs Fuel Canister + 1 dossier).\n  (3,1) Wall      — Impassable.\n  (4,0) Open      — Start.\n  (4,1) Informant — Dr. Vapnik. Gives USB Drive quest.\n  (4,4) Jungle    — Trap! Avoid.\n\nHOW TO EARN 5 DOSSIERS:\n  1. Caches at (0,0) and (0,4): move there, call collect(). (+2)\n  2. USB delivery: talk to Dr. Vapnik at (4,1) about \'jobs\'. He gives a USB Drive.\n     Carry it to Agent Dropout at (2,2) and mention it. (+2)\n  3. Destroy robot: collect Fuel Canister from (2,0), craft Flamethrower at (2,4)\n     for 1 dossier, then move into (0,2). (+3, net +2 after craft cost)\n\nTOOLS:\n  TOOL: move(direction="north|south|east|west")\n  TOOL: talk(message="text")\n  TOOL: collect()\n  TOOL: fabricate(item="Flamethrower")\n\nthink_llm signature: think_llm(operative, world, history, client) -> str\nhistory roles: "observation", "action", "result"\nMust return a string with exactly one TOOL: line.\n\nHOW YOU SHOULD RESPOND:\nReturn ONLY a single Python code block (```python ... ```) containing the complete\nimproved `think_llm` function. The student pastes it directly into their notebook.'


def _analyze_log(log_path: str) -> list:
    try:
        with open(log_path) as f:
            log = _json.load(f)
    except Exception:
        return ["(Could not read game log — run play_mini_mission first.)"]
    turns = [t for t in log.get("turns", []) if "action" in t]
    if not turns:
        return ["(No turns recorded in the log.)"]
    bullets = []
    # Stuck loop
    max_run, cur_run, stuck_action = 1, 1, None
    for i in range(1, len(turns)):
        if turns[i]["action"] == turns[i-1]["action"]:
            cur_run += 1
            if cur_run > max_run:
                max_run, stuck_action = cur_run, turns[i]["action"]
        else:
            cur_run = 1
    if max_run >= 3:
        bullets.append(f"Repeated the same action ({stuck_action}) {max_run} times in a row — stuck in a loop.")
    # Unvisited
    visited = {tuple(t['position']) for t in turns}
    pct = round(len(visited) / 25 * 100)
    if pct < 60:
        bullets.append(f"Only explored {len(visited)}/25 cells ({pct}%) — explore more of the map.")
    # Never talked
    if not any("talk" in t.get("action","").lower() for t in turns):
        bullets.append("Never used talk() — informants give critical quests and items.")
    # Robot bounce
    bounces = sum(1 for t in turns if any(w in t.get("result","").lower() for w in ["bounces back","1 damage"]) and "move" in t.get("action","").lower())
    if bounces:
        bullets.append(f"Bounced off a robot {bounces} time(s) without the right weapon — learn robot weaknesses from informants first.")
    # Dossier shortfall
    final_d = log.get("final_dossiers", 0)
    if final_d < 5:
        gap = 5 - final_d
        bullets.append(f"Finished with {final_d}/5 dossiers — still needed {gap} more. Prioritize deliveries and robot takedowns.")
    return bullets[:5]


def _get_think_llm_source() -> str:
    try:
        return inspect.getsource(think_llm)
    except Exception:
        return "# (source unavailable)"


def generate_debrief_paste(result: dict) -> str:
    global _debrief_initialized
    outcome = "MISSION COMPLETE ✓" if result.get("won") else "MISSION FAILED ✗"
    bullets = _analyze_log(result.get("log_file", ""))
    bullet_block = "\n".join(f"  • {b}" for b in bullets)
    dynamic = (
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "LAST RUN RESULT\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"Outcome : {outcome}\n"
        f"Dossiers: {result.get('dossiers', 0)}/5\n"
        f"Turns   : {result.get('turns', 0)}/50\n"
        f"Health  : {result.get('health', 0)}/3\n"
        "\nWHAT WENT WRONG:\n"
        f"{bullet_block}\n"
        "\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "CURRENT think_llm CELL\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"{_get_think_llm_source()}\n"
    )
    if not _debrief_initialized:
        paste = _STATIC_CONTEXT + "\n" + dynamic
        _debrief_initialized = True
    else:
        paste = dynamic
    return paste


print("Debrief helpers loaded ✓")


In [ ]:
# ── Debrief — run this after every mission run ─────────────────────────────────────
try:
    result
except NameError:
    print("⚠️  Run the mission cell first, then re-run this cell.")
    raise SystemExit()

import ipywidgets as _widgets
from IPython.display import display as _display, HTML as _HTML

paste_text = generate_debrief_paste(result)
is_first = _STATIC_CONTEXT in paste_text
instruction = (
    "📋 FIRST TIME — paste into a NEW conversation with ChatGPT or Gemini."
    if is_first else
    "📋 FOLLOW-UP — paste into the SAME conversation."
)

_escaped = (paste_text
    .replace("\\", "\\\\")
    .replace("`", "\\`")
    .replace("$", "\\$"))

_btn = _widgets.Button(
    description="Copy to Clipboard",
    button_style="info",
    icon="clipboard",
    layout=_widgets.Layout(width="220px", height="36px"),
)
_status = _widgets.Label(value="")

def _on_copy(_):
    _display(_HTML(f"""<script>
(function() {{
  const text = `{_escaped}`;
  if (navigator.clipboard) {{
    navigator.clipboard.writeText(text).catch(() => _fallback(text));
  }} else {{ _fallback(text); }}
  function _fallback(t) {{
    const el = document.createElement('textarea');
    el.value = t; document.body.appendChild(el);
    el.select(); document.execCommand('copy');
    document.body.removeChild(el);
  }}
}})();
</script>"""))
    _status.value = "✓ Copied!"

_btn.on_click(_on_copy)

_display(_HTML(
    f'<div style="font-family:monospace;background:#0a1a0a;color:#00ff41;'
    f'padding:8px 12px;border-radius:6px;border:1px solid #1a4a1a;margin-bottom:6px;">'
    f'<b>DEBRIEF READY</b> — {instruction}</div>'
))
_display(_widgets.HBox([_btn, _status]))
_display(_widgets.Textarea(
    value=paste_text,
    layout=_widgets.Layout(width="100%", height="260px"),
    disabled=True,
))

In [ ]:
# @title Download the game log for debugging
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")

---
## What's Next?

Once your agent can complete this training mission, you're ready for the real challenge:

**Full Mission** (`the_hidden_layer_full_mission.ipynb`) — 8×8 grid, 10 dossiers, 3 informants, 2 robots, 100 turns.